# Activity 004: DNA Detective
In this final activity, we will put together everything we learned in the previous lessons and activities. We will use the 3 techniques we've learned to identify a piece of mystery DNA by comparing it to a database of known organims' DNA.

Remember, the main techniques that we covered are:
1. Identifying and comparing genomic signaturesusing: GC content and k-mers.

2. Comparing the sequences directly using Global Sequence Alignment with a prescribed scoring scheme.

In [ ]:
%pip install biopython

## Grab the functions from previous activities

In [ ]:
import numpy as np
import sys
from Bio import SeqIO
from google.colab import drive
from collections import defaultdict
drive.mount('/content/drive')
sys.path.append("/content/drive/MyDrive/UofT/CS_Academy/")  # REPLACE path to project directory

#============================
# FASTA file Functions
from src import get_fasta_info, get_dna_seq

#============================
# GC content Functions
from src import get_nucleotide_count, get_GC_content, print_gc_content

#============================
# k-mer Functions
from src import count_kmers, get_kmer_percentage, print_kmer_count

#============================
# Hamming Distance
from src import get_hamming_distance

#============================
# Global Sequence Alignment
from src import get_alignment

# 1. Create the DNA database

In [ ]:
base_path = "/content/drive/MyDrive/UofT/CS_Academy/data"  # REPLACE path to data directory

actino_fasta_path = f"{base_path}/actinobacteria_dna.fasta"
cat_fasta_path = f"{base_path}/cat_dna.fasta"
dog_fasta_path = f"{base_path}/dog_dna.fasta"
ecoli_fasta_path = f"{base_path}/e_coli_dna.fasta"
fruit_fly_fasta_path = f"{base_path}/fruit_fly_dna.fasta"
horse_fasta_path = f"{base_path}/horse_dna.fasta"
human_fasta_path = f"{base_path}/human_dna.fasta"
mouse_fasta_path = f"{base_path}/mouse_dna.fasta"
plasmodium_fasta_path = f"{base_path}/plasmodium_falciparum_dna.fasta"
thale_cress_fasta_path = f"{base_path}/thale_cress_dna.fasta"

# Store the DNA sequences
actino_dna = get_dna_seq(actino_fasta_path)
cat_dna = get_dna_seq(cat_fasta_path)
dog_dna = get_dna_seq(dog_fasta_path)
ecoli_dna = get_dna_seq(ecoli_fasta_path)
fruit_fly_dna = get_dna_seq(fruit_fly_fasta_path)
horse_dna = get_dna_seq(horse_fasta_path)
human_dna = get_dna_seq(human_fasta_path)
mouse_dna = get_dna_seq(mouse_fasta_path)
plasmodium_dna = get_dna_seq(plasmodium_fasta_path)
thale_cress_dna = get_dna_seq(thale_cress_fasta_path)

# Set the DNA database
dna_database = {
    "Actinobacteria": actino_dna,
    "Cat": cat_dna,
    "Dog": dog_dna,
    "E. Coli": ecoli_dna,
    "Fruit Fly": fruit_fly_dna,
    "Horse": horse_dna,
    "Human": human_dna,
    "Mouse": mouse_dna,
    "Plasmodium Falciparum": plasmodium_dna,
    "Thale Cress": thale_cress_dna,
}

mystery_fasta_path = f"{base_path}/mystery_dna.fasta"
mystery_dna = get_dna_seq(mystery_fasta_path)

# 2. Compare the GC contents

In [ ]:
print(f"================= GC Contents: =================")

print_gc_content(mystery_dna, "Mystery DNA")

for organism, dna_seq in dna_database.items():
    # TO DO: reuse one of the functions from Activity 000
    # to calculate and print out the GC contents for each
    # organism

# 3. k-mer Comparisons
You can first try looking at specific k-mers and their frequency in the mystery DNA vs the other DNA sequences

In [ ]:
print(f"================= k-mer Counts: =================")
k = 2       # We set the length k that we want to use
kmer = "AT" # Looking for this k-mer count

mystery_len = len(mystery_dna)
comparison_len = min(100, mystery_len)

mystery_kmers, mystery_total_kmer_count = count_kmers(mystery_dna[:comparison_len], k)
print_kmer_count(get_kmer_percentage(mystery_kmers[kmer], mystery_total_kmer_count),
                 "Mystery DNA",
                 f"k-mer: '{kmer}'")

for organism, dna_seq in dna_database.items():
    dna_len = len(dna_seq)
    comparison_len = min(100, mystery_len, dna_len)
    kmers, total_kmer_count = count_kmers(dna_seq[:comparison_len], k)
    print_kmer_count(get_kmer_percentage(kmers[kmer], total_kmer_count),
                     organism,
                     f"k-mer: '{kmer}'")

Next, you can try looking for the k-mer that appears most frequently in the mystery DNA and how often it appears in the other DNA sequences.

<b>Bonus:</b>
It might appear that the percentages for the other DNA sequences are significantly smaller than for the myster DNA. What does this tell you about the length of the mystery DNA compared to the other sequences?

In [ ]:
k = 3   # We set the length k that we want to use

comparison_len = min(100, mystery_len)

mystery_kmers, mystery_total_kmer_count = count_kmers(mystery_dna[:comparison_len], k)

# Grab kmer that appears most frequently in mystery DNA
max_mystery_kmer, max_mystery_kmer_count = max(mystery_kmers.items(), key=lambda item: item[1])
print_kmer_count(get_kmer_percentage(max_mystery_kmer_count, mystery_total_kmer_count),
                 "Mystery DNA",
                 f"k-mer: '{max_mystery_kmer}'")

for organism, dna_seq in dna_database.items():
    dna_len = len(dna_seq)
    comparison_len = min(100, mystery_len, dna_len)
    kmers, total_kmer_count = count_kmers(dna_seq[:comparison_len], k)
    print_kmer_count(get_kmer_percentage(kmers[max_mystery_kmer], total_kmer_count),
                     organism,
                     f"k-mer: '{max_mystery_kmer}'")

# 4. Compare Sequence Alignment Scores
Since the sequences are very long, it would take too long to align them entirely. Instead, we take a substring of each sequence using `comparison_len`.

By default, `comparison_len` is set to `DEFAULT_LEN` ( = 100), unless the mystery or other DNA sequence happens to be smaller than that. The subtring we take starts from the first character in the sequence and ends at index `comparison_len` (in other words, we take the first `comparison_len` # of characters).

Feel free to experiment with:
- `DEFAULT_LEN` to change how long the substring you take is
- Where you take the substring from (hint: instead of taking the first X# of characters, you can take the last X# of characters)

In [ ]:
# Scoring Scheme:
GAP = "-"
MATCH = 1
MISMATCH = -1
GAP_PENALTY = -1

print(f"================= Global Alignment Scores: =================")

DEFAULT_LEN = 100 # Try changing this to get different alignments

for organism, dna_seq in dna_database.items():
    dna_len = len(dna_seq)
    comparison_len = min(DEFAULT_LEN, mystery_len, dna_len)
    S_aligned, T_aligned, max_score = get_alignment(mystery_dna[:comparison_len], dna_seq[:comparison_len])

    print(f"{organism}: {max_score}")
    print(f"{S_aligned}")
    print(f"{T_aligned}\n")

# 5. Conclusion
Now, based on all of the results you gathered, which organism do you think the mystery DNA comes from?

Make sure to thoroughly explain your experiments and reasoning!